In [1]:
#instsll requiered libraries
!pip install -q transformers datasets scikit-learn pandas matplotlib seaborn


In [2]:
!pip install torch 


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
!pip install scikit-learn


In [6]:
from datasets import load_dataset
# Load IMDB dataset directly
dataset = load_dataset("imdb")

# Convert training data into DataFrame
df = pd.DataFrame(dataset['train'])

# Show first 5 rows
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [7]:
#load the dataset from tensorflow
from datasets import load_dataset
dataset = load_dataset("imdb")

In [8]:
#explore the dataset structure
print(f"Training samples: {len(dataset['train'])}")
print(f"Testing samples: {len(dataset['test'])}")

Training samples: 25000
Testing samples: 25000


In [10]:
#look at a sample review and its label
sample_review = dataset['train'][0]
print(f"Review: {sample_review['label']} (0=Negative, 1=Positive)")
print(f"Review text (first 500 characters):\n{sample_review['text'][:500]}")


Review: 0 (0=Negative, 1=Positive)
Review text (first 500 characters):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent


In [11]:
#check class balance
train_labels = [dataset['train'][i]['label'] for i in range(len(dataset['train']))]
unique, counts = np.unique(train_labels, return_counts=True)
print(f"Negative reviews: {counts[0]}:,)({counts[0]/len(train_labels)*100:.1f}%)")
print(f"Positive reviews: {counts[1]}:,)({counts[1]/len(train_labels)*100:.1f}%)")


Negative reviews: 12500:,)(50.0%)
Positive reviews: 12500:,)(50.0%)


In [12]:
#create a smaller subset for quick experimentation
train_subset  = dataset['train'].shuffle(seed=42).select(range(5000))
test_subset = dataset['test'].shuffle(seed=42).select(range(1000))

print(f"Subset training samples: {len(train_subset)}")
print(f"Subset testing samples: {len(test_subset)}")

Subset training samples: 5000
Subset testing samples: 1000


In [17]:
#text preprocessing - cleaning the text

def clean_text(text):
    """
    clean and preprocess text for analysis
    """ 
    #lowercase
    text = text.lower()
    
    #remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    #remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    #remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text




In [18]:
import re
sample_text = train_subset[0]['text']

sample_label = "Positive" if train_subset[0]['label'] == 1 else "Negative"

cleaned_sample = clean_text(sample_text)

print("Label:", sample_label)

print("\nCleaned Review:\n")

print(cleaned_sample[:500])

Label: Positive

Cleaned Review:

there is no relation at all between fortier and profiler but the fact that both are police series about violent crimes profiler looks crispy fortier looks classic profiler plots are quite simple fortiers plot are far more complicated fortier looks more like prime suspect if we have to spot similarities the main character is weak and weirdo but have clairvoyance people like to compare to judge to evaluate how about just enjoying funny thing too people writing fortier looks american but on the oth


In [20]:
# Apply cleaning to the entire subset

train_text_clean = [clean_text(item['text']) for item in train_subset]

test_text_clean = [clean_text(item['text']) for item in test_subset]

# Extract labels

train_labels = [item['label'] for item in train_subset]

test_labels = [item['label'] for item in test_subset]

In [21]:
!pip install -q spacy

In [23]:
#load the small English model
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy loaded successfully!")

spaCy loaded successfully!


In [24]:
#analyze the sample review with spacy
sample_review = train_text_clean[0]
doc = nlp(sample_review)
print(f"Analyzing review: {sample_review[:200]}...")


Analyzing review: there is no relation at all between fortier and profiler but the fact that both are police series about violent crimes profiler looks crispy fortier looks classic profiler plots are quite simple forti...


In [25]:
#tokens the building blocks of text
tokens = [token.text for token in doc]

In [27]:
# Part of Speech grammatical roles

pos_counts = {}

for token in doc:
    pos_counts[token.pos_] = pos_counts.get(token.pos_, 0) + 1

for pos, count in sorted(pos_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{pos}: {count} occurrences")

NOUN: 28 occurrences
VERB: 17 occurrences
ADJ: 17 occurrences
ADV: 13 occurrences
DET: 10 occurrences
CCONJ: 8 occurrences
ADP: 7 occurrences
AUX: 7 occurrences
PRON: 6 occurrences
PART: 5 occurrences
SCONJ: 3 occurrences
PROPN: 3 occurrences


In [28]:
#named entities Recognized - finding people , places
entities = [(ent.text, ent.label_) for ent in doc.ents][:10]  # Show first 10 entities
if entities:
    for entity in entities:
        print(f"Entity: {entity[0]} - {entity[1]}")
else:
    print("No named entities found in the review.")

Entity: american - NORP
Entity: american - NORP
Entity: english - LANGUAGE
Entity: american - NORP


In [29]:
#Sentiment of individual words using spacy
positive_words = ["good", "bad", "excellent", "terrible", "amazing", "awful"]
negative_words = ["bad", "terrible", "awful", "horrible", "worst", "disappointing"]

count_positive = [w for w in positive_words if w in sample_review.lower()]
count_negative = [w for w in negative_words if w in sample_review.lower()]

print(f"positive words found :{count_positive if count_positive else 'None'}")
print(f"negative words found :{count_negative if count_negative else 'None'}")

print(f"\n INSIGHT: Text is structured! words have grammatical roles")


positive words found :['good']
negative words found :None

 INSIGHT: Text is structured! words have grammatical roles


In [30]:
#convert the text to another
from sklearn.feature_extraction.text import TfidfVectorizer
#create tf-idf vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # Limit to top 5000 features
    stop_words='english',
    ngram_range=(1,2)
)
print("Converting text to TF-IDF")

x_train_tfidf = tfidf_vectorizer.fit_transform(train_text_clean)
x_test_tfidf = tfidf_vectorizer.fit_transform(test_text_clean)
y_train = np.array(train_labels)
y_test = np.array(test_labels)


Converting text to TF-IDF


In [31]:
print(f"Training matrix shape: {x_train_tfidf.shape}")
print(f"Testing matrix shape: {x_test_tfidf.shape}")
print(f"{x_test_tfidf.shape[0]} reviews x {x_test_tfidf.shape[1]} features")

Training matrix shape: (5000, 5000)
Testing matrix shape: (1000, 5000)
1000 reviews x 5000 features


In [32]:
#what words are most important 
feature_names = tfidf_vectorizer.get_feature_names_out()
print("These words help distinguish one review from another")


These words help distinguish one review from another


In [33]:
#show top 20 features
print("Top 20 features:{feature_names[:20].tolist()}")


Top 20 features:{feature_names[:20].tolist()}


In [34]:
#find the highest scoring feature for a sample review
review_index = 0
review_score = x_test_tfidf[review_index].toarray()[0]
top_feature_index =review_score.argsort()[-10:][::-1]  # Get indices of top 10 features
top_features = [(feature_names[index] , review_score[index] )
                for index in top_feature_index if review_score[index] > 0][:10]

In [37]:
review_features = cleaned_sample.split()
print(f"\nTop features for the sample review: {review_features[:10]}")


Top features for the sample review: ['there', 'is', 'no', 'relation', 'at', 'all', 'between', 'fortier', 'and', 'profiler']


In [39]:
from sklearn.linear_model import LogisticRegression

# Train Logistic Regression model

classifier = LogisticRegression(max_iter=1000, random_state=42)

classifier.fit(x_train_tfidf, y_train)

print("Model trained successfully!")

Model trained successfully!


In [41]:
from sklearn.metrics import accuracy_score
y_pred = classifier.predict(x_test_tfidf)

#calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.53


In [43]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


              precision    recall  f1-score   support

    Negative       0.56      0.40      0.47       512
    Positive       0.52      0.67      0.58       488

    accuracy                           0.53      1000
   macro avg       0.54      0.53      0.52      1000
weighted avg       0.54      0.53      0.52      1000



In [44]:
#get feature coefficients to understand which words are most influential in the model's predictions
coefficients = classifier.coef_[0]
feature_names = tfidf_vectorizer.get_feature_names_out()

#get top positive and negative features
top_positive_idx = coefficients.argsort()[-10:][::-1]  # Indices of top 10 positive features
top_negative_idx = coefficients.argsort()[:10]  # Indices of top 10

In [47]:
for idx in top_positive_idx[:15]:
    print(f"{feature_names[idx]}: {coefficients[idx]:.3f}")
    

for idx in top_negative_idx[:15]:
    print(f"{feature_names[idx]}: {coefficients[idx]:.3f}")


given: 4.618
bicycle: 3.499
episodes: 3.197
lets: 2.654
women: 2.441
faithful: 2.300
amateurish: 2.270
food: 2.231
people dont: 2.148
elses: 2.117
balance: -4.867
worst: -4.373
watch: -3.203
political: -3.035
brave: -2.778
bad: -2.751
investigate: -2.698
theatrical: -2.541
ponyo: -2.366
scientist: -2.282


In [48]:
def predict_review(review_text, model, vectorizer):
    """
    Predict the sentiment of a review using the trained logistic regression model.
    """
    # Clean the input review
    cleaned_review = clean_text(review_index)
    
    # Convert to TF-IDF features
    review_tfidf = tfidf_vectorizer.transform([cleaned_review])
    
    # Predict sentiment
    prediction = classifier.predict(review_tfidf)[0]
    
    probability = model.predict_proba(vectorizer)
    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probability[1] if prediction == 1 else probability[0]
    
    return sentiment, confidence

In [49]:
#test some reviews
test_reviews = [
    "This movie was fantastic! I loved it.",
    "Terrible film. Waste of time.",
    "It was okay, not great but not bad either.",
    "An absolute masterpiece. Highly recommend!",
    "Worst movie I've ever seen."
]


In [ ]:
for review in test_reviews:

    sentiment, confidence = predict_review(
        review,
        classifier,
        tfidf_vectorizer
    )

    print(f"Review: {review[:60]}")
    print(f"Sentiment: {sentiment} (Confidence: {confidence:.1%})")
    print("-" * 50)

Review: This movie was fantastic! I loved it.
Sentiment: Negative (Confidence: 52.8%)
--------------------------------------------------
Review: Terrible film. Waste of time.
Sentiment: Negative (Confidence: 53.4%)
--------------------------------------------------
Review: It was okay, not great but not bad either.
Sentiment: Negative (Confidence: 81.7%)
--------------------------------------------------
Review: An absolute masterpiece. Highly recommend!
Sentiment: Positive (Confidence: 52.1%)
--------------------------------------------------
Review: Worst movie I've ever seen.
Sentiment: Negative (Confidence: 72.3%)
--------------------------------------------------
